In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import gc
import ast
import copy
import time
import random
import numpy as np
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as torchdata

from Model.MENDR.MENDREncoder import MENDRAutoEncoder, WaveletEncoderDecoder
from Model.MENDR.MENDRContextualizer import MENDRContextualizer
from Model.MENDR.MENDRTrainer import MENDRTrainer
from Model.MENDR.mAtt.optimizer import MixOptimizer
from Model.MENDR.mAtt.spd import SPDTangentSpace

from dataset import WaveletFinetuningDataset
from torch_geometric.loader import DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, auc
import optuna
from optuna.trial import TrialState

from torchmetrics.classification import BinaryAccuracy, MulticlassAccuracy

/home/hice1/mchen439/scratch/miniconda3/envs/Wavelet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load Dataset
print("*" * 50)
train_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/train")
print("Dataset Loaded. Length of Train Dataset: ", len(train_dataset))
eval_dataset = WaveletFinetuningDataset(root="/home/hice1/mchen439/scratch/downstreamTaskData/eval")
print("Dataset Loaded. Length of Validation Dataset: ", len(eval_dataset))


**************************************************
Loading 2130 folders


100%|██████████| 2130/2130 [01:42<00:00, 20.86it/s]


Dataset Loaded. Length of Train Dataset:  44854
Loading 253 folders


100%|██████████| 253/253 [00:10<00:00, 25.22it/s]

Dataset Loaded. Length of Validation Dataset:  5101


In [11]:
### Seed ###
torch.cuda.empty_cache()
random.seed(42)
os.environ['PYTHONHASHSEED'] = str(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
## CUDNN ##
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [12]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
train_loader = DataLoader(train_dataset, batch_size=128, num_workers=2, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=128, num_workers=2, shuffle=True) # It shouldn't shuffle but we do it so that there are normal and abnormals in each batch
print(len(train_loader), len(eval_loader))

351 40


In [13]:
### Model ###
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = MENDRAutoEncoder(device=device)
contextualizer = MENDRContextualizer(device=device)

'''
for param in encoder.parameters():
    param.requires_grad = False

for param in contextualizer.parameters():
    param.requires_grad = False
'''

encoder.load_state_dict(torch.load("checkpoint/batch_size256_40epochs/encoder_weights.pth"))
contextualizer.load_state_dict(torch.load("checkpoint/batch_size256_40epochs/contextualizer_weights.pth"))

#optim_params = []
#optim_params += list(encoder.parameters())
#optim_params += list(contextualizer.parameters())
#optim_params += list(decoder.parameters())

#optimizer = torch.optim.AdamW(optim_params, lr=0.001)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, threshold=.1, mode='min')
#optimizer = MixOptimizer(optimizer)


<All keys matched successfully>

In [14]:
def _std_norm(x):
	mean = torch.mean(x, dim=(0, 1), keepdim=True)
	std = torch.std(x, dim=(0, 1), keepdim=True)
	x = (x - mean) / std
	return x

In [19]:
def train_one_epoch(encoder, contextualizer, decoder, optimizer, task_loss_fn, train_loader):
	loss_sum = 0
	encoder.train()
	contextualizer.train()
	decoder.train()

	tangent_space = SPDTangentSpace(19, device)
	flatten = nn.Flatten().to(device)

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	balanced_accuracies = []

	BinaryAcc = BinaryAccuracy().to(device)
	BalancedAcc = MulticlassAccuracy(2, average='macro').to(device)

	for iteration in pbar:
		data = next(data_iterator)

		for key, value in data.items():
			if isinstance(value, torch.Tensor):
				# Perform batch normalization across channels 
				data[key] = _std_norm(value.float().to(device))

		relevant_bands = [data['delta'].float().to(device),
		 				  data['theta'].float().to(device),
						  data['alpha'].float().to(device),
						  data['beta'].float().to(device),
						  data['gamma'].float().to(device)]

		inputs = dict(zip(BANDS, relevant_bands))

		encoder_outputs = encoder(data['graph'], inputs)
		wavelet_manifold_output, epoched_shape = contextualizer.WaveletContextualizer(encoder_outputs)
		_, combined_manifold_output = contextualizer.CombinedContextualizer(wavelet_manifold_output, epoched_shape)

		combined_r2e_output = flatten(tangent_space(combined_manifold_output))
		embed_cat = [combined_r2e_output]
		for band, embedding in wavelet_manifold_output.items():
			embedding_r2e = tangent_space(embedding)
			embedding_r2e = flatten(embedding_r2e)
			embed_cat.append(embedding_r2e)
		embed_cat = torch.cat(embed_cat, dim=1)
		'''
		combined_r2e_output, combined_manifold_output, wavelet_manifold_output = contextualizer(encoder_outputs)
		embed_cat = [combined_r2e_output]
		for band, embedding in wavelet_manifold_output.items():
			embedding_r2e = tangent_space(embedding).view(combined_r2e_output.view(1024, 4, -1))
			embedding_r2e = embedding_r2e.flatten()
			embed_cat.append(embedding)
		embed_cat = torch.cat(embed_cat, dim=1)
		'''

		prediction = decoder(embed_cat).float()
		ground_truth = data['graph'].y.to(device).float()
		ground_truth = torch.repeat_interleave(ground_truth, 4)

		#assert prediction.shape[0] == ground_truth.shape[0]

		task_loss = task_loss_fn(prediction, ground_truth[:, None])

		with torch.no_grad():
			score_y = torch.sigmoid(prediction)
			pred_y = torch.gt(score_y, 0.5).long().cpu().numpy()
			score_y = score_y.cpu().numpy()

			balanced_accuracy = balanced_accuracy_score(ground_truth.cpu().numpy(), pred_y)
			accuracy = accuracy_score(ground_truth.cpu().numpy(), pred_y)
			roc_auc = roc_auc_score(ground_truth.cpu().numpy(), score_y)
			precision, recall, thresholds = precision_recall_curve(ground_truth.cpu().numpy(), score_y, pos_label=1)
			pr_auc = auc(recall, precision)

		#print(torch.argmax(prediction, axis=1).shape)
		#accuracy = BinaryAcc(prediction, ground_truth)
		#balanced_accuracy = BalancedAcc(prediction, ground_truth)
		balanced_accuracies.append(balanced_accuracy)

		loss = task_loss

		optimizer.scheduler_step(iteration)

		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
		loss_sum += loss.item()

		pbar.set_postfix(task_loss=task_loss.item(), accuracy=f'{accuracy*100:.3f}', balanced_accuracy=f'{balanced_accuracy*100:.3f}', AUROC=f'{roc_auc*100:.3f}', AUCPR=f'{pr_auc*100:.3f}', lr=optimizer.scheduler.get_last_lr()[0])
	balanced_accuracies = np.array(balanced_accuracies)	
	print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
	return loss_sum

In [22]:
def validate_one_epoch(encoder, contextualizer, decoder, task_loss_fn, val_loader):
	loss_sum = 0
	encoder.eval()
	contextualizer.eval()
	decoder.eval()

	tangent_space = SPDTangentSpace(19, device)
	flatten = nn.Flatten().to(device)

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	balanced_accuracies = []

	BinaryAcc = BinaryAccuracy().to(device)
	BalancedAcc = MulticlassAccuracy(2, average='macro').to(device)

	with torch.no_grad():
		for iteration in pbar:
			data = next(data_iterator)
			
			for key, value in data.items():
				if isinstance(value, torch.Tensor):
					# Perform batch normalization across channels 
					data[key] = _std_norm(value.float().to(device))

			relevant_bands = [data['delta'].float().to(device),
							data['theta'].float().to(device),
							data['alpha'].float().to(device),
							data['beta'].float().to(device),
							data['gamma'].float().to(device)]

			inputs = dict(zip(BANDS, relevant_bands))

			encoder_outputs = encoder(data['graph'], inputs)
			combined_r2e_output, combined_manifold_output, wavelet_manifold_output = contextualizer(encoder_outputs)

			wavelet_manifold_output, epoched_shape = contextualizer.WaveletContextualizer(encoder_outputs)
			_, combined_manifold_output = contextualizer.CombinedContextualizer(wavelet_manifold_output, epoched_shape)

			combined_r2e_output = flatten(tangent_space(combined_manifold_output))
			embed_cat = [combined_r2e_output]
			for band, embedding in wavelet_manifold_output.items():
				embedding_r2e = tangent_space(embedding)
				embedding_r2e = flatten(embedding_r2e)
				embed_cat.append(embedding_r2e)
			embed_cat = torch.cat(embed_cat, dim=1)

			prediction = decoder(embed_cat).float()
			ground_truth = data['graph'].y.to(device).float()
			ground_truth = torch.repeat_interleave(ground_truth, 4)


			task_loss = task_loss_fn(prediction, ground_truth[:, None])

			loss = task_loss
			loss_sum += loss.item()

			#correct = (torch.argmax(prediction, axis=1) == ground_truth).sum().item()
			#accuracy = correct / prediction.size(0)
			with torch.no_grad():
				score_y = torch.sigmoid(prediction)
				pred_y = torch.gt(score_y, 0.5).long().cpu().numpy()

				balanced_accuracy = balanced_accuracy_score(ground_truth.cpu().numpy(), pred_y)
				accuracy = accuracy_score(ground_truth.cpu().numpy(), pred_y)


			#accuracy = BinaryAcc(prediction, ground_truth)
			#balanced_accuracy = BalancedAcc(prediction, ground_truth)
			balanced_accuracies.append(balanced_accuracy)

			pbar.set_postfix(
				task_loss=task_loss.item(),
				accuracy=f'{accuracy*100:.3f}',
				balanced_accuracy=f'{balanced_accuracy*100:.3f}')

		balanced_accuracies = np.array(balanced_accuracies)
		print(f'Mean Acc: {balanced_accuracies.mean()} Acc Std: {balanced_accuracies.std()}')
		return loss_sum, balanced_accuracies.mean(), balanced_accuracies.std()

In [17]:
class optimalModel(nn.Module):
    def __init__(self):
        super(optimalModel, self).__init__()
        #self.layer0 = nn.Sequential(nn.Linear(16*10*200, 10*200), nn.Dropout(p=0.1), nn.GELU())
        #self.layer0 = nn.Sequential(nn.Linear(6*760 + 7300, 1520), nn.Dropout(p=0.1), nn.GELU())
        #self.layer02 = nn.Sequential(nn.Linear(27740, 730), nn.Dropout(p=0.1), nn.GELU())
        #self.layer1 = nn.Sequential(nn.Linear(1520, 190), nn.Dropout(p=0.1), nn.GELU())
        #self.fusion_layer = nn.Sequential(nn.Bilinear(190, 1460 // 4, 190), nn.Dropout(p=0.1), nn.GELU())
        self.layer0 =  nn.Sequential(nn.Linear(1140, 190), nn.Dropout(p=0.1), nn.GELU())
        self.out = nn.Sequential(nn.Linear(190, 1))

        '''
        self.encoder_processor = nn.ParameterDict({
            #'delta': nn.Sequential(nn.Flatten()),
            #'theta': nn.Sequential(nn.Flatten()),
            #'alpha': nn.Sequential(nn.Flatten()),
            #'beta':  nn.Sequential(nn.Flatten()),
            #'gamma': nn.Sequential(nn.Flatten())
            'delta': nn.Sequential(nn.Conv1d(19, 10, 2, stride=2), nn.Flatten()),
            'theta': nn.Sequential(nn.Conv1d(19, 10, 2, stride=2), nn.Flatten()),
            'alpha': nn.Sequential(nn.Conv1d(19, 10, 2, stride=2), nn.Flatten()),
            'beta':  nn.Sequential(nn.Conv1d(19, 10, 2, stride=2), nn.Flatten()),
            'gamma': nn.Sequential(nn.Conv1d(19, 10, 2, stride=2), nn.Flatten())
        })
        '''

    def forward(self, x):
        '''
        encoding_features = []
        for band, encoding in encoding.items():
            band_out = self.encoder_processor[band](encoding[0])
            encoding_features.append(band_out)
        encoding_features = torch.cat(encoding_features, dim=1)
        x = torch.cat([x, encoding_features], dim=1)
        '''
        x = self.layer0(x)
        #x2 = self.layer02(encoding_features)
        #x = self.layer1(x)
        #x = self.layer2(x)
        return self.out(x)
        #return x

decoder = optimalModel().to(device)
#optimizer = optim.Adam(decoder.parameters(), lr=0.001)
optim_params = []
optim_params.append({'params': encoder.parameters(), 'lr': 1e-5})
optim_params.append({'params': contextualizer.parameters(), 'lr': 1e-4})
optim_params.append({'params': decoder.parameters(), 'lr': 1e-3})
contextualizer.CombinedContextualizer.combined_spd_transform2 = nn.Sequential()
optimizer = torch.optim.AdamW(optim_params, lr=1e-3)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=50, threshold=.1, mode='min')
optimizer = MixOptimizer(optimizer)
optimizer.set_scheduler_t0(176)
##scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=44, T_mult=2)
print("Total Params:", sum(p.numel() for p in encoder.parameters() if p.requires_grad))
print("Total Params:", sum(p.numel() for p in contextualizer.parameters() if p.requires_grad))
print("Total Params:", sum(p.numel() for p in decoder.parameters() if p.requires_grad))

Set Cosine Annealing with Warm Restarts optimizer T_0 to: 176
Total Params: 216981


In [23]:
for epoch_idx in range(8):
    training_loss = train_one_epoch(encoder, contextualizer, decoder, optimizer, nn.BCEWithLogitsLoss(), train_loader)
    eval_loss, eval_acc, eval_std = validate_one_epoch(encoder, contextualizer, decoder, nn.BCEWithLogitsLoss(), eval_loader)
    print("*" * 50)
    print(f"Epoch: {epoch_idx}")
    print("Training Loss:", training_loss)
    #print(f"Eval Loss: {eval_loss} Eval Mean Acc: {eval_acc} Eval Std: {eval_std}")

Training: 100%|██████████| 351/351 [23:40<00:00,  4.05s/it, AUCPR=78.118, AUROC=81.619, accuracy=67.130, balanced_accuracy=66.587, lr=5.58e-6, task_loss=0.562]


Mean Acc: 0.6531952111615734 Acc Std: 0.05834697081261275


Validating: 100%|██████████| 40/40 [02:16<00:00,  3.41s/it, accuracy=68.578, balanced_accuracy=68.319, task_loss=0.608]


Mean Acc: 0.6812105247123655 Acc Std: 0.034521046019522356
**************************************************
Epoch: 0
Training Loss: 213.7740667462349


Training: 100%|██████████| 351/351 [23:47<00:00,  4.07s/it, AUCPR=85.830, AUROC=87.769, accuracy=73.148, balanced_accuracy=71.979, lr=5.58e-6, task_loss=0.486]


...

Mean Acc: 0.6674503847052845 Acc Std: 0.053992480259862606


Validating: 100%|██████████| 40/40 [02:11<00:00,  3.29s/it, accuracy=61.239, balanced_accuracy=60.343, task_loss=0.686]


Mean Acc: 0.6908483659046615 Acc Std: 0.03634774789930433
**************************************************
Epoch: 2
Training Loss: 208.69955295324326


Training:   1%|          | 4/351 [00:17<25:37,  4.43s/it, AUCPR=78.409, AUROC=77.246, accuracy=69.336, balanced_accuracy=68.635, lr=9.99e-6, task_loss=0.563]


KeyboardInterrupt: 